# Prototype


In [1]:
import importlib
import evaluate as evaluate_module
import analysis as analysis_module
from evaluate import evaluate, print_metrics, print_evaluation_summary
from benchmarks import load_benchmarks
from llm import call_llm, build_prompt
from analysis import analyze
import output as output_module

Loading of the benchmarks.


In [2]:
# Configuration: change these variables as needed
from pathlib import Path
BENCHMARK_DIR = Path("./benchmark")
BENCHMARK_TYPES = ["socbenchd_1"]  # add more benchmark types here
BENCHMARK_LIMIT = 1   # set to None for all sectors (these are e.g. socbenchd_1)
QUERY_LIMIT = 3       # set to None for all queries

benchmark_sets, total_available, total_queries = load_benchmarks(BENCHMARK_DIR, BENCHMARK_TYPES, BENCHMARK_LIMIT, QUERY_LIMIT)
print(f"Loaded {len(benchmark_sets)} benchmark sets (total available: {total_available})")
print(f"Total queries: {total_queries}")



Loaded 1 benchmark sets (total available: 11)
Total queries: 3


In [3]:
# Editable prompt template: modify this cell to change the instruction given to the LLM.
PROMPT_TEMPLATE = '''You're doing a Service Composition.
You are given a set of REST API specifications and a task description.
Your job is to write Python code using the appropriate client library that fulfills the task by calling the necessary endpoints in the correct order. 

Rules:
- Use the requests library.
- Only use endpoints defined in the provided specifications.
- Return ONLY raw Python code. No markdown, no code fences, no comments, no notes, no explanations — nothing but the code itself.
- All the code shall be under a function called compose.

source
{services_block}

## Task

{query}

## Python Code
'''


In [ ]:
# Call the LLM for all benchmark queries in a single sector
MODEL = "nvidia/Nemotron-3-Nano-Omni"

if benchmark_sets:
    benchmark = benchmark_sets[0]
    sector_results = []
    for query_index, query in enumerate(benchmark['queries'], start=1):
        prompt = build_prompt(benchmark['services'], query['query'], PROMPT_TEMPLATE)
        generated = call_llm(prompt, MODEL, '')
        generated += '\n\ncompose()'
        # save services and metadata for reproducibility
        sector_results.append({
            'query_index': query_index,
            'query': query,
            'generated': generated,
            'service_files': benchmark.get('service_files', []),
            'model': MODEL
        })
        print(f"--- Query {query_index}: {query['query']} ---")
        print('Generated code:\n', generated)


--- Query 1: Retrieve the status and performance metrics of all monitored energy equipment, access active alerts for any system performance issues, analyze the impact of weather conditions on electricity demand between specific dates, configure new alert thresholds for unusual energy generation patterns in specific sectors, and submit the integration status of renewable energy sources such as solar or wind into the energy grid. ---
Generated code:
 import requests

def compose():
    # Retrieve the status and performance metrics of all monitored energy equipment
    equipment_status = requests.get("https://api.energyinsights.com/v1/equipment-monitoring").json()
    
    # Access active alerts for any system performance issues
    active_alerts = requests.get("https://api.energysector.com/alerts").json()
    
    # Analyze the impact of weather conditions on electricity demand between specific dates
    weather_impact_analysis = requests.get("https://api.energysector.com/weather-impact-

In [5]:
# Evaluate the original generated code for every query in the selected sector
if benchmark_sets:
    for result in sector_results:
        initial_metrics = evaluate(result['generated'], result['query'].get('endpoints', []))
        result['initial_metrics'] = initial_metrics
        print_metrics(initial_metrics, f"Initial evaluation - Query {result['query_index']}")


Initial evaluation - Query 1
  Precision: 0.80
  Recall:    0.80
  F1:        0.80
  Extracted: ['GET /alerts', 'GET /equipment-monitoring', 'GET /weather-impact-analysis', 'POST /alert-settings', 'POST /renewable/integration/status']
  Expected:  ['GET /alerts', 'GET /equipment-status', 'GET /weather-impact-analysis', 'POST /alert-settings', 'POST /renewable/integration/status']
  Missing:   ['GET /equipment-status']
  Extra:     ['GET /equipment-monitoring']
Initial evaluation - Query 2
  Precision: 1.00
  Recall:    1.00
  F1:        1.00
  Extracted: ['GET /carbon-emissions', 'GET /prediction-summary', 'GET /resources/status', 'POST /report-feedback', 'POST /smart-meters/data']
  Expected:  ['GET /carbon-emissions', 'GET /prediction-summary', 'GET /resources/status', 'POST /report-feedback', 'POST /smart-meters/data']
  Missing:   []
  Extra:     []
Initial evaluation - Query 3
  Precision: 0.86
  Recall:    0.75
  F1:        0.80
  Extracted: ['GET /energy-patterns', 'GET /real-ti

In [6]:
# Analyze the generated code with a Python linter helper for every query in the selected sector
if benchmark_sets:
    for result in sector_results:
        analysis = analyze(result['generated'])
        result['analysis'] = analysis
        print(f"Analysis - Query {result['query_index']}\n", analysis)


Analysis - Query 1
 AST: No syntax errors found.
Ruff: F841 Local variable `equipment_status` is assigned to but never used
Ruff:  --> /var/folders/57/p3z5bqsd11l937yr1409c3m00000gn/T/tmpoz7gu42b/generated.py:5:5
Ruff:   |
Ruff: 3 | def compose():
Ruff: 4 |     # Retrieve the status and performance metrics of all monitored energy equipment
Ruff: 5 |     equipment_status = requests.get("https://api.energyinsights.com/v1/equipment-monitoring").json()
Ruff:   |     ^^^^^^^^^^^^^^^^
Ruff: 6 |     
Ruff: 7 |     # Access active alerts for any system performance issues
Ruff:   |
Ruff: help: Remove assignment to unused variable `equipment_status`
Ruff: F841 Local variable `active_alerts` is assigned to but never used
Ruff:   --> /var/folders/57/p3z5bqsd11l937yr1409c3m00000gn/T/tmpoz7gu42b/generated.py:8:5
Ruff:    |
Ruff:  7 |     # Access active alerts for any system performance issues
Ruff:  8 |     active_alerts = requests.get("https://api.energysector.com/alerts").json()
Ruff:    |     ^^

In [7]:
# Run the LLM again using the analysis and the original code for every query in the selected sector
if benchmark_sets:
    for result in sector_results:
        refined_prompt = f'''You are given the original task, a review of the generated code, and the original code.
Use the review to improve the code.

## Task
{result['query']['query']}

## Analysis
{result['analysis']}

## Original Code
{result['generated']}

Return ONLY raw Python code. No markdown, no code fences, no comments, no notes, no explanations — nothing but the code itself.
'''
        generated_refined = call_llm(refined_prompt, MODEL, 'Return only Python code, no explanation.')
        result['generated_refined'] = generated_refined
        print(f"Refined code - Query {result['query_index']}:\n", generated_refined)


Refined code - Query 1:
 import requests

def compose():
    requests.get("https://api.energyinsights.com/v1/equipment-monitoring").json()
    
    requests.get("https://api.energysector.com/alerts").json()
    
    weather_impact_analysis = requests.get("https://api.energysector.com/weather-impact-analysis?start_date=2023-10-01&end_date=2023-10-31").json()
    
    alerts_configured = requests.post("https://api.energyinsights.com/v1/alert-settings", json={
        "threshold": 1000,
        "sector": "Industrial",
        "notificationType": "email"
    })
    
    renewable_integration = requests.post("https://api.energysector.com/renewable/integration/status", json={
        "source": "solar",
        "capacity": 5000,
        "integration_status": "integrated"
    })
    
    return weather_impact_analysis, alerts_configured, renewable_integration

compose()
Refined code - Query 2:
 import requests

def compose():
    # Submit data from smart meters installed in the facility
    re

In [8]:
# Final evaluation of the refined output for every query in the selected sector
for result in sector_results:
    refined_metrics = evaluate(result['generated_refined'], result['query'].get('endpoints', []))
    result['refined_metrics'] = refined_metrics
    print_metrics(refined_metrics, f"Refined evaluation - Query {result['query_index']}")




Refined evaluation - Query 1
  Precision: 0.80
  Recall:    0.80
  F1:        0.80
  Extracted: ['GET /alerts', 'GET /equipment-monitoring', 'GET /weather-impact-analysis', 'POST /alert-settings', 'POST /renewable/integration/status']
  Expected:  ['GET /alerts', 'GET /equipment-status', 'GET /weather-impact-analysis', 'POST /alert-settings', 'POST /renewable/integration/status']
  Missing:   ['GET /equipment-status']
  Extra:     ['GET /equipment-monitoring']
Refined evaluation - Query 2
  Precision: 1.00
  Recall:    1.00
  F1:        1.00
  Extracted: ['GET /carbon-emissions', 'GET /prediction-summary', 'GET /resources/status', 'POST /report-feedback', 'POST /smart-meters/data']
  Expected:  ['GET /carbon-emissions', 'GET /prediction-summary', 'GET /resources/status', 'POST /report-feedback', 'POST /smart-meters/data']
  Missing:   []
  Extra:     []
Refined evaluation - Query 3
  Precision: 0.86
  Recall:    0.75
  F1:        0.80
  Extracted: ['GET /energy-patterns', 'GET /real-ti

In [9]:
# Save outputs to disk under output/<date>_<name>
from datetime import datetime
run_name = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
# ensure we're using the latest output module implementation
importlib.reload(output_module)
outdir = output_module.make_output_dir(run_name)
for r in sector_results:
    output_module.write_query_output(outdir, r)
output_module.write_overall_summary(outdir, sector_results)
print('Wrote outputs to', outdir)


Wrote outputs to output/2026-06-01_17-02-24


In [10]:
print_evaluation_summary([result['initial_metrics'] for result in sector_results], "Initial Evaluation Summary")
print_evaluation_summary([result['refined_metrics'] for result in sector_results], "Refined Evaluation Summary")


Initial Evaluation Summary
  Average Precision: 0.89
  Average Recall:    0.85
  Average F1:        0.87
  Avg. Missing Endpoints: 1.00
  Avg. Extra Endpoints:   0.67
  Correct Compositions: 1/3 (33.3%)
Refined Evaluation Summary
  Average Precision: 0.89
  Average Recall:    0.85
  Average F1:        0.87
  Avg. Missing Endpoints: 1.00
  Avg. Extra Endpoints:   0.67
  Correct Compositions: 1/3 (33.3%)
